# Immersed finite element bases for Stokes interface problems

This notebook builds the local basis of an **immersed finite element (IFE)** for 2D Stokes flow,
on a reference triangle cut by an interface $\Gamma$. On each side of $\Gamma$ the basis is an
ordinary polynomial, and the two sides are glued together by the physical interface conditions.
Two element pairs are built:

- **P2–P1 (Taylor–Hood):** quadratic velocity, linear pressure.
- **Stabilized P1–P1:** linear velocity, linear pressure. It needs a pressure-stabilization term
  in the global weak form; that term does not change the basis.

For each pair: the constraints, the construction, plots of every shape function, and a check of
the delta property.

## The problem

On each side $s\in\{a,b\}$ of $\Gamma$, Stokes flow with viscosity $\mu_s$:

$$
-\nabla\cdot\boldsymbol\sigma^s=\mathbf g,\qquad \nabla\cdot\mathbf u^s=0,\qquad
\boldsymbol\sigma^s=-p^s\mathbf I+\mu_s\big(\nabla\mathbf u^s+(\nabla\mathbf u^s)^T\big).
$$

Across $\Gamma$, with $[q]=q^b-q^a$ and $\mathbf n$ the unit normal from side $a$ into side $b$:

$$
[\mathbf u]=\mathbf J\ (\text{constant}),\qquad [\boldsymbol\sigma\mathbf n]=\mathbf j_t=-\mathbf f ,
$$

where $\mathbf f$ is the force per unit length that the interface exerts on the fluid. Exact
solutions also satisfy $[\nabla\cdot\mathbf u]=0$ and, for a continuous body force,
$[\mu\Delta\mathbf u-\nabla p]=\mathbf 0$.

## The cut cell

- **Reference triangle.** Vertices $0:(0,0)$, $1:(0,1)$, $2:(1,0)$; edge midpoints $3:(0,\tfrac12)$,
  $4:(\tfrac12,0)$, $5:(\tfrac12,\tfrac12)$.
- **Interface.** $\Gamma$ is the segment from $(x_0,0)$ to $(0,y_0)$, with normal $\mathbf n\propto(y_0,x_0)$.
- **Sides.** $K_a$ is the small triangle containing vertex 0; $K_b$ is the rest.
- **Ownership.** Each node belongs to the side it lies in. Its value is read from that side's polynomial.

## How the basis is computed

1. **Unknowns.** On each side, $u$ and $v$ are polynomials of degree $k$ ($k=2$ or $1$) and $p$
   is linear. Their coefficients are the unknowns $\mathbf c$.
2. **Data.** The nodal values $U_j,V_j,P_j$, plus the interface data $\mathbf J$ and $\mathbf j_t$.
   Together they form $\mathbf d$.
3. **Conditions.** All of them are linear, so they form the system $A\mathbf c=B\mathbf d$. Each row
   says "a combination of coefficients = one data item, or 0", and $B$ records which data item
   feeds each row.
4. **Solve.** $C=A^{-1}B$. Column $j$ of $C$ is the function whose data item $j$ is 1 and all
   others 0:
   - the nodal columns are the **shape functions**;
   - the interface columns are **correction functions**, which carry an interface force or jump.

| condition | P2–P1 rows | P1–P1 rows |
|---|---|---|
| **nodal:** $u,v$ at the velocity nodes, $p$ at the vertices, from the owning side | $6+6+3=15$ | $3+3+3=9$ |
| **continuity:** $[\mathbf u]=\mathbf J$ at $k+1$ points of $\Gamma$ | 6 | 4 |
| **traction:** $[\boldsymbol\sigma\mathbf n]=\mathbf j_t$ at $k$ points of $\Gamma$ | 4 | 2 |
| **divergence:** $[\nabla\cdot\mathbf u]=0$, every coefficient | 3 | 1 |
| **momentum:** $[\mu\Delta\mathbf u-\nabla p]=\mathbf 0$ | 2 | 2 |
| **total** | **30** | **18** |

On a straight $\Gamma$ each condition is a polynomial identity along the line, and the chosen
number of points fixes it exactly.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from types import SimpleNamespace

x, y = sp.symbols('x y')
half = sp.Rational(1, 2)
VERTICES = [(0, 0), (0, 1), (1, 0)]
MIDPOINTS = [(0, half), (half, 0), (half, half)]


def cut_cell_system(k, X0, Y0, mua, mub):
    # The linear system A c = B d of the P_k-P1 IFE (k = 2: Taylor-Hood, k = 1: P1-P1) on the reference
    # triangle cut by the segment from (X0, 0) to (0, Y0), with viscosity mua on K_a and mub on K_b.
    mono = [1, x, y] if k == 1 else [1, x, y, x*y, x**2, y**2]
    vnodes = VERTICES + (MIDPOINTS if k == 2 else [])

    # unknowns: on each side, u and v of degree k, p linear
    Q, unknowns = {}, []
    for s in 'ab':
        cu, cv, cp = sp.symbols(f'u{s}0:{len(mono)}'), sp.symbols(f'v{s}0:{len(mono)}'), sp.symbols(f'p{s}0:3')
        Q[s] = (sum(c*m for c, m in zip(cu, mono)), sum(c*m for c, m in zip(cv, mono)), cp[0] + cp[1]*x + cp[2]*y)
        unknowns += [*cu, *cv, *cp]

    # data: nodal values, velocity jump, traction jump at k points of Gamma
    names = ([f'U{j}' for j in range(len(vnodes))] + [f'V{j}' for j in range(len(vnodes))]
             + ['P0', 'P1', 'P2', 'Ju', 'Jv'] + [f'jt{j}{c}' for j in range(k) for c in 'xy'])
    d = dict(zip(names, sp.symbols(names)))

    # geometry
    n = np.array([Y0, X0]) / np.hypot(X0, Y0)                     # unit normal, K_a -> K_b
    owner = lambda P: 'a' if P[0]/X0 + P[1]/Y0 < 1 else 'b'
    at = lambda f, P: f.subs({x: P[0], y: P[1]})
    ends, mid = [(X0, 0), (0, Y0)], (X0/2, Y0/2)
    cont_pts, trac_pts = (ends, [mid]) if k == 1 else (ends + [mid], ends)
    mu = {'a': mua, 'b': mub}

    def traction(s, P):                                             # sigma n = -p n + mu (grad u + grad u^T) n
        u, v, p = Q[s]
        G = sp.Matrix([[u.diff(x), u.diff(y)], [v.diff(x), v.diff(y)]])
        return [at(t, P) for t in (-p*sp.eye(2) + mu[s]*(G + G.T)) * sp.Matrix(n)]

    rows = []
    # 1. nodal values, read from the side that owns the node
    for i, (kind, nodes) in enumerate([('U', vnodes), ('V', vnodes), ('P', VERTICES)]):
        rows += [at(Q[owner(P)][i], P) - d[f'{kind}{j}'] for j, P in enumerate(nodes)]
    # 2. continuity [u] = J at k+1 points of Gamma
    rows += [at(Q['b'][i] - Q['a'][i], P) - d[J] for i, J in ((0, 'Ju'), (1, 'Jv')) for P in cont_pts]
    # 3. traction [sigma n] = jt at k points of Gamma
    for j, P in enumerate(trac_pts):
        ta, tb = traction('a', P), traction('b', P)
        rows += [tb[c] - ta[c] - d[f'jt{j}{"xy"[c]}'] for c in (0, 1)]
    # 4. divergence [div u] = 0, every coefficient of the polynomial
    div = {s: Q[s][0].diff(x) + Q[s][1].diff(y) for s in 'ab'}
    rows += sp.Poly(sp.expand(div['b'] - div['a']), x, y).coeffs()
    # 5. momentum [mu Lap u - grad p] = 0
    res = lambda s, i: mu[s]*(Q[s][i].diff(x, 2) + Q[s][i].diff(y, 2)) - Q[s][2].diff([x, y][i])
    rows += [res('b', i) - res('a', i) for i in (0, 1)]

    A, b = sp.linear_eq_to_matrix(rows, unknowns)
    A = np.array(A.tolist(), float)
    B = np.array(b.jacobian(list(d.values())).tolist(), float)
    return SimpleNamespace(k=k, A=A, B=B, Q=Q, unknowns=unknowns, names=names, vnodes=vnodes, mono=mono)


def build_basis(k, X0, Y0, mua, mub):
    # C = A^-1 B. Returns {data name: {side: (u, v, p)}} for every column of C, and the velocity nodes.
    cc = cut_cell_system(k, X0, Y0, mua, mub)
    C = np.linalg.solve(cc.A, cc.B)
    rank = np.linalg.matrix_rank(cc.A)
    print(f"P{k}-P1: {len(cc.A)} conditions, {len(cc.unknowns)} unknowns, rank A = {rank}")
    assert len(cc.A) == len(cc.unknowns) == rank, "the cut-cell system must be square and nonsingular"
    return {name: {s: [f.subs(dict(zip(cc.unknowns, C[:, j]))) for f in cc.Q[s]] for s in 'ab'}
            for j, name in enumerate(cc.names)}, cc.vnodes


def plot_panels(items, X0, Y0, ncols, n=30):
    # items: list of (function {side: (u, v, p)}, component 0/1/2, title). K_a red, K_b green, Gamma blue.
    nrows = -(-len(items) // ncols)
    pts = np.array([(i/n, j/n) for i in range(n + 1) for j in range(n + 1 - i)]
                   + [(X0*t, Y0*(1 - t)) for t in np.linspace(0, 1, n)])
    level = pts[:, 0]/X0 + pts[:, 1]/Y0
    fig = plt.figure(figsize=(3.4*ncols, 2.9*nrows), dpi=60)
    for k, (F, comp, title) in enumerate(items):
        ax = fig.add_subplot(nrows, ncols, k + 1, projection='3d')
        for s, color, mask in (('a', 'r', level <= 1 + 1e-9), ('b', 'g', level >= 1 - 1e-9)):
            f = sp.lambdify((x, y), F[s][comp], 'numpy')
            z = np.broadcast_to(f(pts[mask, 0], pts[mask, 1]), mask.sum()).astype(float)
            ax.plot_trisurf(pts[mask, 0], pts[mask, 1], z, color=color, alpha=0.75, linewidth=0, shade=False)
        ax.plot([X0, 0], [0, Y0], [0, 0], color='b', lw=2)
        ax.view_init(elev=22, azim=-50)
        ax.set_title(title)
    plt.tight_layout()
    plt.show()


def plot_basis(basis, X0, Y0):
    # one panel per shape function, in the component it interpolates (u for U_j, v for V_j, p for P_j);
    # one row per kind: U, V, P
    names = [nm for nm in basis if nm[0] in 'UVP']
    plot_panels([(basis[nm], 'UVP'.index(nm[0]), nm) for nm in names], X0, Y0,
                ncols=sum(nm[0] == 'U' for nm in names))


def delta_matrix(basis, vnodes, X0, Y0):
    # N[i, j] = shape function j at DOF i, read from the side that owns the node. Delta property: N = I.
    owner = lambda P: 'a' if P[0]/X0 + P[1]/Y0 < 1 else 'b'
    dofs = [nm for nm in basis if nm[0] in 'UVP']
    node = lambda nm: (vnodes if nm[0] in 'UV' else VERTICES)[int(nm[1:])]
    N = np.zeros((len(dofs), len(dofs)))
    for i, dof in enumerate(dofs):
        P = node(dof)
        for j, name in enumerate(dofs):
            N[i, j] = float(basis[name][owner(P)]['UVP'.index(dof[0])].subs({x: P[0], y: P[1]}))
    return N, dofs


def print_delta(N, dofs):
    print(f"max |N - I| = {np.abs(N - np.eye(len(dofs))).max():.1e}")
    print("     " + "".join(f"{d:>4s}" for d in dofs))
    for d, row in zip(dofs, np.round(N, 10) + 0.0):
        print(f"{d:>4s} " + "".join(f"{v:4.0f}" for v in row))
    assert np.abs(N - np.eye(len(dofs))).max() < 1e-10, "delta property fails: N is not the identity"

## Parameters

The cut and the two viscosities are set once here and used everywhere below. Change them and re-run the notebook.

In [ ]:
X0, Y0 = 0.62, 0.37        # Gamma runs from (X0, 0) to (0, Y0)
MUA, MUB = 1.0, 10.0        # viscosity on K_a and on K_b

---
# P2–P1 (Taylor–Hood)

## Constraints

- **Unknowns.** Per side, $u$ and $v$ are quadratic ($1,x,y,xy,x^2,y^2$) and $p$ is linear: $2\times(6+6+3)=30$.
- **DOFs.** $u,v$ at the 3 vertices and 3 edge midpoints, and $p$ at the 3 vertices: 15.
- **Interface data.** $\mathbf J$, plus $\mathbf j_t$ at the two ends of $\Gamma$.

Why each group has the size it has:
- **Continuity at 3 points.** The velocity jump is quadratic along $\Gamma$; 3 points fix it.
- **Traction at the 2 ends.** $\nabla\mathbf u$ and $p$ are linear, so the traction jump is linear along $\Gamma$; 2 points fix it.
- **Divergence: 3 rows.** $[\nabla\cdot\mathbf u]$ is linear, and all three coefficients are set
  to zero. That covers its value along $\Gamma$ and its normal derivative. Divergence is **not**
  set to zero on each side: incompressibility is enforced weakly by the global weak form, and a
  standard Taylor–Hood shape function is not divergence-free either.
- **Momentum: 2 rows.** $\Delta\mathbf u$ and $\nabla p$ are constant on each side, so this is one constant vector.

## Construction

In [ ]:
basis2, vnodes2 = build_basis(2, X0, Y0, MUA, MUB)

## Visualization: all 15 shape functions

One panel per shape function, drawn in the component it interpolates.
- **Top row:** $U_0$–$U_5$ (the 3 vertices, then the 3 midpoints), drawn in $u$.
- **Middle row:** $V_0$–$V_5$, drawn in $v$.
- **Bottom row:** $P_0$–$P_2$, drawn in $p$.

What to look for:
- **Velocity shape functions** are continuous across $\Gamma$, with a kink set by the viscosity ratio.
- **Pressure shape functions** are the plain P1 hats.

In [ ]:
plot_basis(basis2, X0, Y0)

## Delta property

Each shape function must be 1 at its own DOF (its own field, at its own node) and 0 at every
other DOF, in every field. $N_{ij}$ is shape function $j$ evaluated at DOF $i$, read from the side
that owns the node. $N$ must be the identity.

In [ ]:
print_delta(*delta_matrix(basis2, vnodes2, X0, Y0))

---
# Stabilized P1–P1

## Constraints

- **Unknowns.** Per side, $u$, $v$ and $p$ are linear: $2\times(3+3+3)=18$.
- **DOFs.** $u,v,p$ at the 3 vertices: 9.
- **Interface data.** $\mathbf J$, plus **one** traction jump $\mathbf j_t$, at the midpoint of $\Gamma$.

The five groups shrink:
- **Continuity:** at the two ends of $\Gamma$, since a linear trace is fixed by 2 points.
- **Divergence:** $\nabla\cdot\mathbf u$ is constant on each side, so 1 row.
- **Momentum:** $\Delta\mathbf u=0$, so the momentum row reads $[\nabla p]=\mathbf 0$.
- **Traction:** one point is enough. The velocity gradient is constant on each side, and $[p]$ is
  constant along $\Gamma$, so the traction jump is the same everywhere on $\Gamma$.

## Construction

In [ ]:
basis1, vnodes1 = build_basis(1, X0, Y0, MUA, MUB)

## Visualization: all 9 shape functions

- **Top row:** $U_0$–$U_2$, drawn in $u$.
- **Middle row:** $V_0$–$V_2$, drawn in $v$.
- **Bottom row:** $P_0$–$P_2$, drawn in $p$.

The pieces are linear, so each one is a flat plane.

In [ ]:
plot_basis(basis1, X0, Y0)

## Delta property

In [ ]:
print_delta(*delta_matrix(basis1, vnodes1, X0, Y0))

---
# Appendix: additional checks

These checks go beyond the delta property:
- **Coupling:** what else a shape function carries.
- **Structure:** reduction to the standard element when there is no interface physics, and partition of unity.
- **Exact solutions:** exact interface solutions must be rebuilt exactly.
- **Pressure jumps:** the pressure-jump formulas, and the role of the interface data in the pressure step.
- **Conditioning**, and a **known issue** of P2–P1.

The helpers below use `cut_cell_system` from the construction code.

In [ ]:
def solve_C(cc):
    return np.linalg.solve(cc.A, cc.B)


def geometry(X0, Y0):
    n = np.array([Y0, X0]) / np.hypot(X0, Y0)
    return n, np.array([-n[1], n[0]]), (lambda P: 'a' if P[0]/X0 + P[1]/Y0 < 1 else 'b')


def fields(cc, coef):
    # per-side polynomials (u, v, p) of a coefficient vector
    return {s: [f.subs(dict(zip(cc.unknowns, coef))) for f in cc.Q[s]] for s in 'ab'}


def coefficients(cc, F):
    # coefficient vector of piecewise fields F = {side: (u, v, p)}, in the layout of cc.unknowns
    return np.array([float(sp.Poly(sp.expand(f), x, y).coeff_monomial(m))
                     for s in 'ab' for f, mono in zip(F[s], (cc.mono, cc.mono, [1, x, y])) for m in mono])


def traction_of(F, mu, n, P):
    u, v, p = (sp.sympify(f) for f in F)
    G = sp.Matrix([[u.diff(x), u.diff(y)], [v.diff(x), v.diff(y)]])
    return np.array([float(t.subs({x: P[0], y: P[1]})) for t in (-p*sp.eye(2) + mu*(G + G.T)) * sp.Matrix(n)])


def data_vector(cc, F, X0, Y0, mua, mub):
    # nodal values (owner side) and interface data of piecewise fields F
    n, _, owner = geometry(X0, Y0)
    val = lambda f, P: float(sp.sympify(f).subs({x: P[0], y: P[1]}))
    d = {f'{kind}{j}': val(F[owner(P)][i], P)
         for i, (kind, nodes) in enumerate((('U', cc.vnodes), ('V', cc.vnodes), ('P', VERTICES)))
         for j, P in enumerate(nodes)}
    d['Ju'], d['Jv'] = (val(F['b'][i] - F['a'][i], (X0, 0)) for i in (0, 1))
    trac_pts = [(X0/2, Y0/2)] if cc.k == 1 else [(X0, 0), (0, Y0)]
    for j, P in enumerate(trac_pts):
        d[f'jt{j}x'], d[f'jt{j}y'] = traction_of(F['b'], mub, n, P) - traction_of(F['a'], mua, n, P)
    return np.array([d[name] for name in cc.names])


def pressure_jump(cc, coef, P):
    F = fields(cc, coef)
    return float((F['b'][2] - F['a'][2]).subs({x: P[0], y: P[1]}))


L = [1 - x - y, y, x]                                      # P1 hats at vertices 0, 1, 2
STANDARD = {1: L, 2: [l*(2*l - 1) for l in L] + [4*L[0]*L[1], 4*L[0]*L[2], 4*L[1]*L[2]]}


def check_structure(k, X0, Y0, mua, mub):
    # equal viscosities on both sides (no interface physics) vs. the given pair
    cc_eq, cc = cut_cell_system(k, X0, Y0, mua, mua), cut_cell_system(k, X0, Y0, mua, mub)
    C_eq, C = solve_C(cc_eq), solve_C(cc)
    def standard(name):
        f = [0, 0, 0]
        f['UVP'.index(name[0])] = (L if name[0] == 'P' else STANDARD[k])[int(name[1:])]
        return coefficients(cc, {'a': f, 'b': f})
    dofs = [nm for nm in cc.names if nm[0] in 'UVP']
    col = lambda C_, nm: C_[:, cc.names.index(nm)]
    ok = all(np.allclose(col(C_eq, nm), standard(nm)) for nm in dofs)
    print(f"mu_a = mu_b = {mua}: every shape function is the standard one:", ok)
    assert ok, "equal viscosities must give the standard shape functions"
    ok = all(np.allclose(col(C, f'P{j}'), standard(f'P{j}')) for j in range(3))
    print(f"mu = ({mua}, {mub}): pressure shape functions are plain P1 hats:", ok)
    assert ok, "the pressure shape functions must be plain P1 hats"
    for kind, target in (('U', [1, 0, 0]), ('V', [0, 1, 0]), ('P', [0, 0, 1])):
        total = sum(col(C, nm) for nm in dofs if nm[0] == kind)
        ok = np.allclose(total, coefficients(cc, {'a': target, 'b': target}))
        print(f"{kind} shape functions sum to (u, v, p) = {tuple(target)}:", ok)
        assert ok, f"the {kind} shape functions must sum to {tuple(target)}"


def linear_family(X0, Y0, mua, mub, rng):
    # kinked shear along Gamma + linear divergence-free field + velocity jump + pressure step + pressure gradient
    n, t, _ = geometry(X0, Y0)
    s0, ca, cb, G, p0, Pi = rng.normal(size=6)
    J, a0, m = rng.normal(size=2), rng.normal(size=2), rng.normal(size=3)
    M = [[m[0], m[1]], [m[2], -m[0]]]
    xi = n[0]*(x - X0) + n[1]*y                            # signed distance from Gamma
    F = {}
    for s, c in (('a', ca), ('b', cb)):
        j = J if s == 'b' else (0, 0)
        F[s] = ((s0 + c*xi)*t[0] + a0[0] + M[0][0]*x + M[0][1]*y + j[0],
                (s0 + c*xi)*t[1] + a0[1] + M[1][0]*x + M[1][1]*y + j[1],
                G*(t[0]*x + t[1]*y) + p0 + (Pi if s == 'b' else 0))
    return F


def poiseuille_family(X0, Y0, mua, mub, rng):
    # parabolic flow along Gamma, a different profile on each side
    n, t, _ = geometry(X0, Y0)
    s0, ca, cb, G, p0, Pi = rng.normal(size=6)
    J = rng.normal(size=2)
    xi = n[0]*(x - X0) + n[1]*y
    F = {}
    for s, c, mu in (('a', ca, mua), ('b', cb, mub)):
        prof, j = s0 + c*xi + G*xi**2/(2*mu), (J if s == 'b' else (0, 0))
        F[s] = (prof*t[0] + j[0], prof*t[1] + j[1], G*(t[0]*x + t[1]*y) + p0 + (Pi if s == 'b' else 0))
    return F


def stream_family(X0, Y0, mua, mub, rng):
    # u = (psi_y, -psi_x), psi cubic, the same on both sides; grad p = mu Lap u on each side
    q = rng.normal(size=9)
    psi = (q[0]*x + q[1]*y + q[2]*x**2 + q[3]*x*y + q[4]*y**2
           + q[5]*x**3 + q[6]*x**2*y + q[7]*x*y**2 + q[8]*y**3)
    u, v = psi.diff(y), -psi.diff(x)
    lap = sp.expand(psi.diff(x, 2) + psi.diff(y, 2))
    return {s: (u, v, mu*(lap.diff(y)*x - lap.diff(x)*y)) for s, mu in (('a', mua), ('b', mub))}


def check_exact(k, families, cuts, ratios, trials=2):
    rng = np.random.default_rng(0)
    for name, family in families.items():
        err = 0.0
        for X0, Y0 in cuts:
            for mua, mub in ratios:
                cc = cut_cell_system(k, X0, Y0, mua, mub)
                C = solve_C(cc)
                for _ in range(trials):
                    F = family(X0, Y0, mua, mub, rng)
                    exact = coefficients(cc, F)
                    err = max(err, np.abs(C @ data_vector(cc, F, X0, Y0, mua, mub) - exact).max() / max(1, np.abs(exact).max()))
        print(f"{name:>16s}: max relative coefficient error over {len(cuts)} cuts x {len(ratios)} viscosity pairs = {err:.1e}")
        assert err < 1e-8, f"{name} solutions are not rebuilt exactly"


def check_balloon(k, X0, Y0, mua, mub):
    # u = 0, p = 0 on K_a and 1 on K_b: rebuilt from nodal data only, and with the interface data
    F = {'a': (0, 0, 0), 'b': (0, 0, 1.0)}
    cc = cut_cell_system(k, X0, Y0, mua, mub)
    C, d = solve_C(cc), data_vector(cc, F, X0, Y0, mua, mub)
    nodal = np.array([nm[0] in 'UVP' for nm in cc.names])
    exact = coefficients(cc, F)
    rebuilt = np.allclose(C @ d, exact)
    print(f"mu = ({mua}, {mub}):  [p] from nodal data only = {pressure_jump(cc, C @ (d*nodal), (X0/2, Y0/2)):+.4f},  "
          f"with interface data = {pressure_jump(cc, C @ d, (X0/2, Y0/2)):+.4f},  "
          f"rebuilt exactly: {rebuilt}")
    assert rebuilt and abs(pressure_jump(cc, C @ d, (X0/2, Y0/2)) - 1) < 1e-10, \
        "with the interface data the pressure step must be rebuilt exactly"
    assert not np.allclose(C @ (d*nodal), exact), "nodal values alone should not rebuild the pressure step"


def conditioning(k, cuts, ratios):
    # condition number of A after scaling every row to unit max-norm (removes the units of mu)
    print("equilibrated cond(A)  " + "".join(f"  mu_b/mu_a={mb/ma:<7g}" for ma, mb in ratios))
    for X0, Y0 in cuts:
        conds = [np.linalg.cond((A := cut_cell_system(k, X0, Y0, ma, mb).A) / np.abs(A).max(axis=1, keepdims=True))
                 for ma, mb in ratios]
        print(f"  cut ({X0:<5}, {Y0:<5})   " + "".join(f"  {c:<17.1e}" for c in conds))


def singular_ratios(k, X0, Y0):
    # all ratios q = mu_b/mu_a in [1e-6, 1e6] at which A is singular. A is affine in each viscosity,
    # A = A0 + mu_a Aa + mu_b Ab, so with mu_a = 1 it is singular when q solves (A0 + Aa) z = -q Ab z.
    A0 = cut_cell_system(k, X0, Y0, 0, 0).A
    Aa, Ab = cut_cell_system(k, X0, Y0, 1, 0).A - A0, cut_cell_system(k, X0, Y0, 0, 1).A - A0
    w = sl.eigvals(A0 + Aa, -Ab)
    w = w[np.isfinite(w) & (np.abs(w.imag) < 1e-8*np.maximum(1, np.abs(w)))].real
    return np.sort(w[(w > 1e-6) & (w < 1e6)])


def combine(basis, weights):
    # linear combination of functions from build_basis: {name: weight} -> {side: (u, v, p)}
    return {s: [sum(w*basis[nm][s][i] for nm, w in weights.items()) for i in range(3)] for s in 'ab'}


import scipy.linalg as sl
CUTS = [(0.62, 0.37), (0.4, 0.4), (0.05, 0.9), (0.95, 0.1)]
RATIOS = [(1.0, 1.0), (1.0, 10.0), (10.0, 1.0), (1.0, 20.0)]

## P2–P1

### Coupling: what else a shape function carries

The stress balance ties velocity and pressure together across $\Gamma$. So a velocity shape
function is not zero in its other components:
- $U_0$ has a small $v$ component, through the shear stress, when $\mu_a\ne\mu_b$;
- $U_0$ also carries a pressure that jumps across $\Gamma$ by $2[\mu]\,\partial_nu_n$.

The last panel is a **correction function**: the response to a unit tangential force at
$(0,y_0)$, with all nodal values zero. In a global solve, interface forces enter through these
functions, together with the load $\int_\Gamma\mathbf f\cdot\mathbf v\,ds$.

In [ ]:
n, t, _ = geometry(X0, Y0)
plot_panels([(basis2['U0'], 1, r'$v$ of $U_0$'), (basis2['U0'], 2, r'$p$ of $U_0$'),
             (combine(basis2, {'jt1x': t[0], 'jt1y': t[1]}), 0, 'correction: u, tangential force')], X0, Y0, ncols=3)

### Structure

Three properties are checked:
1. **No interface physics.** With $\mu_a=\mu_b$ and no interface data, every shape function equals the standard Taylor–Hood one.
2. **Pressure shape functions.** They are plain P1 hats with zero velocity, for any viscosities.
3. **Partition of unity.** The $U$, $V$ and $P$ shape functions each sum to the matching constant field.

In [ ]:
check_structure(2, X0, Y0, MUA, MUB)

### Exact solutions are rebuilt exactly

Any piecewise polynomial Stokes solution that lies in the element space and satisfies the
interface conditions must be rebuilt exactly from its own nodal values and interface data. The
interface data are computed directly from the solution.

Three families are used:
- **Linear:** a kinked shear along $\Gamma$ plus a linear divergence-free field, a velocity jump and a pressure step.
- **Poiseuille:** a parabolic flow along $\Gamma$, with a different profile on each side.
- **Stream function:** a cubic stream function with a viscosity contrast. Its interface force varies along $\Gamma$.

In [ ]:
check_exact(2, {'linear': linear_family, 'Poiseuille': poiseuille_family, 'stream function': stream_family},
            CUTS, RATIOS)

### A force that varies along $\Gamma$: the pressure jump conditions

Rebuild a stream-function solution with $\mu_a\ne\mu_b$ from its data. Then read the jumps off the
rebuilt polynomials, with $\mathbf f=-[\boldsymbol\sigma\mathbf n]$. They must satisfy

$$
[p]=\hat f_1-2[\mu]\,\boldsymbol\tau\cdot\partial_\tau\mathbf u,\qquad
[\partial_np]=\partial_s\hat f_2+2[\mu]\,\partial_{\tau\tau}u_n .
$$

In [ ]:
cc = cut_cell_system(2, X0, Y0, MUA, MUB)
F = stream_family(X0, Y0, MUA, MUB, np.random.default_rng(3))
G = fields(cc, solve_C(cc) @ data_vector(cc, F, X0, Y0, MUA, MUB))
n, t, _ = geometry(X0, Y0)
ends, dmu, L_gamma = [(X0, 0), (0, Y0)], MUB - MUA, np.hypot(X0, Y0)
f = [traction_of(G['a'], MUA, n, P) - traction_of(G['b'], MUB, n, P) for P in ends]
dd = lambda e, a: a[0]*e.diff(x) + a[1]*e.diff(y)                     # directional derivative
for k, P in enumerate(ends):
    pj = float((G['b'][2] - G['a'][2]).subs({x: P[0], y: P[1]}))
    formula = f[k] @ n - 2*dmu*float((t[0]*dd(G['a'][0], t) + t[1]*dd(G['a'][1], t)).subs({x: P[0], y: P[1]}))
    print(f"end {k}: [p] = {pj:+.6f}   f1 - 2[mu] tau.d_tau u = {formula:+.6f}")
    assert np.isclose(pj, formula, rtol=1e-8, atol=1e-8), f"the [p] jump condition fails at end {k}"
dpn = float(dd(G['b'][2], n) - dd(G['a'][2], n))
ds_f2 = (f[1] @ t - f[0] @ t)/L_gamma                                   # s runs from end 0 to end 1
formula = ds_f2 + 2*dmu*float(dd(dd(n[0]*G['a'][0] + n[1]*G['a'][1], t), t))
print(f"[d_n p] = {dpn:+.6f}   d_s f2 + 2[mu] d_tt u_n = {formula:+.6f}   (d_s f2 = {ds_f2:+.4f})")
assert np.isclose(dpn, formula, rtol=1e-8, atol=1e-8), "the [d_n p] jump condition fails"

### The pressure step needs the interface data

The balloon has $\mathbf u=0$ and a pressure that jumps by 1 across $\Gamma$. It is rebuilt two ways:
- **Nodal values only:** it cannot be exact. When $\mu_a=\mu_b$, a shape function with zero jumps has no pressure step.
- **Nodal values and interface data:** it is exact.

This is why, in a global solve, an interface force must enter through the correction functions and
not only through the load $\int_\Gamma\mathbf f\cdot\mathbf v\,ds$.

In [ ]:
check_balloon(2, X0, Y0, MUA, MUA)    # equal viscosities
check_balloon(2, X0, Y0, MUA, MUB)

### Conditioning across cuts

Each row of $A$ is scaled to unit size first, so the numbers measure the geometry and not the
units of $\mu$. The cuts include the symmetric cut $x_0=y_0$ and cuts close to a vertex.

In [ ]:
conditioning(2, [(0.62, 0.37), (0.4, 0.4), (0.1, 0.1), (0.01, 0.01), (0.99, 0.5)],
             [(1.0, 1.0), (1.0, 10.0), (10.0, 1.0), (1.0, 100.0)])

### Known issue: singular viscosity ratios

For a fixed cut, $A$ is linear in each viscosity: $A=A_0+\mu_aA_a+\mu_bA_b$. So $A$ is singular
exactly when the ratio $q=\mu_b/\mu_a$ is an eigenvalue of the pencil $(A_0+A_a)\,z=-q\,A_b\,z$.

**Finding.** The P2–P1 matrix has isolated singular ratios. At such a ratio a nonzero function
vanishes at all 15 DOFs and satisfies every interface condition, so the nodal values do not
determine the basis. Moderate ratios are safe. This is **open**. The P1–P1 element, built from the
same five groups, has none (checked below).

In [ ]:
r = singular_ratios(2, 0.48, 0.44)
sv = np.linalg.svd(cut_cell_system(2, 0.48, 0.44, 1.0, r[0]).A, compute_uv=False)
print(f"cut (0.48, 0.44): singular at mu_b/mu_a = {r},  sigma_min/sigma_max there = {sv[-1]/sv[0]:.1e}")
grid = np.linspace(0.1, 0.9, 5)
found = np.array([q for a in grid for b in grid for q in singular_ratios(2, a, b)])
print(f"{len(grid)**2} cuts in [0.1, 0.9]^2: {len(found)} singular ratios; none in "
      f"{found[found < 1].max():.3g} < mu_b/mu_a < {found[found > 1].min():.3g}")
# moderate ratios are safe: none of the singular ratios lies in the range the tests above use
assert not np.any((found > 0.05) & (found < 20)), "a singular ratio with 0.05 < mu_b/mu_a < 20"

## Stabilized P1–P1

### Coupling

$U_0$ carries a small $v$ component, and a pressure that is two parallel planes with a constant
step between them. The last panel is the correction function for a unit normal force: a pressure
step with zero velocity.

In [ ]:
n, t, _ = geometry(X0, Y0)
plot_panels([(basis1['U0'], 1, r'$v$ of $U_0$'), (basis1['U0'], 2, r'$p$ of $U_0$'),
             (combine(basis1, {'jt0x': -n[0], 'jt0y': -n[1]}), 2, 'correction: p, normal force')], X0, Y0, ncols=3)

### Structure

In [ ]:
check_structure(1, X0, Y0, MUA, MUB)

### Exact solutions are rebuilt exactly

Only the linear family lies in the P1–P1 space. It is tested up to a 1000:1 viscosity ratio.

In [ ]:
check_exact(1, {'linear': linear_family}, CUTS + [(0.01, 0.01)],
            RATIOS + [(1.0, 1000.0), (1000.0, 1.0)], trials=3)

### The pressure step needs the interface data

In [ ]:
check_balloon(1, X0, Y0, MUA, MUA)    # equal viscosities
check_balloon(1, X0, Y0, MUA, MUB)

### Conditioning across cuts

In [ ]:
conditioning(1, [(0.62, 0.37), (0.4, 0.4), (0.1, 0.1), (0.01, 0.01), (0.99, 0.5)],
             [(1.0, 1.0), (1.0, 10.0), (10.0, 1.0), (1.0, 1000.0)])

### Singular viscosity ratios

The same pencil analysis as for P2–P1, over cuts from 0.01 to 0.99.

In [ ]:
grid = np.r_[0.01, np.linspace(0.05, 0.95, 10), 0.99]
found = [(a, b, q) for a in grid for b in grid for q in singular_ratios(1, a, b)]
print(f"{len(grid)**2} cuts: singular ratios in [1e-6, 1e6]: {found if found else 'none'}")
assert not found, "the P1-P1 system should have no singular viscosity ratio"

### The stabilization term in the global weak form

The basis is only the local space. P1–P1 also needs a pressure-stabilization term in the global
weak form, subtracted from the continuity equation so that it acts on the pressure test function.
Three standard choices:

| option | term | notes |
|---|---|---|
| continuous interior penalty | $\gamma\sum_F \dfrac{h_F^3}{\mu}\int_F[\nabla p\cdot\mathbf n_F][\nabla q\cdot\mathbf n_F]$ over interior mesh faces | weakly consistent; the same form as a pressure ghost penalty, applied on all interior faces |
| Brezzi–Pitkäranta | $\beta\sum_T\sum_\alpha \dfrac{h_T^2}{\mu_\alpha}\int_{T_\alpha}\nabla p\cdot\nabla q$ | simplest; $O(h)$ inconsistent, which is acceptable at first order |
| PSPG | $\delta\sum_T\sum_\alpha \dfrac{h_T^2}{\mu_\alpha}\int_{T_\alpha}(\nabla p-\mathbf g)\cdot\nabla q$ | consistent (the viscous term vanishes for P1) |

Points specific to the immersed element:
- **Per side.** On a cut cell, evaluate the term separately on each side, $T_\alpha=T\cap K_\alpha$, using that side's polynomial.
- **Nothing across $\Gamma$.** No term is needed across $\Gamma$ inside a cell; the basis already has $[\nabla p]=0$ there.
- **It also acts on velocity shape functions.** They carry a pressure part, so the stabilization acts on them too. This is automatic, because it is a term of the coupled form.
- **Correction function.** Its contribution to this term moves to the right-hand side, together with its other terms.

---
# Summary

| | P2–P1 (Taylor–Hood) | stabilized P1–P1 |
|---|---|---|
| local system | 30×30 | 18×18 |
| delta property | yes | yes |
| reduces to the standard element when $\mu_a=\mu_b$ | yes | yes |
| pressure shape functions | plain P1 hats | plain P1 hats |
| exact solutions rebuilt exactly | linear, Poiseuille and stream-function families, including a force varying along $\Gamma$ | linear family, up to 1000:1 viscosity |
| interface force per cut cell | linear along $\Gamma$ (2 traction points) | constant (1 traction point) |
| singular viscosity ratios | **yes**: isolated ratios at large contrast (open) | none found |
| stabilization in the global form | not needed | needed |

**How the basis is used in a global solve, for both pairs:**
- The nodal values are the unknowns, and the nodal columns of $C$ are the shape functions.
- On each cut cell, the interface data multiplied by the interface-data columns of $C$ give a
  correction function, added to the solution. Its stiffness moves to the right-hand side.
- The force also enters the right-hand side as $\int_\Gamma\mathbf f\cdot\mathbf v\,ds$.
- The shape functions do not match across cut edges (as usual for immersed elements), so the global
  form needs consistency and penalty terms on those edges. These terms apply to the correction
  function as well.

**Not covered here:**
- **The element map.** The conditions are written on the reference triangle with reference
  derivatives. On a general physical triangle, the gradients, the normal and the second derivatives
  must be mapped with the element Jacobian before the system is solved.
- **Body-force jumps.** A body force that jumps across $\Gamma$ (gravity with a density jump) needs
  a right-hand side in the momentum row.
- **Global assembly** and an inf-sup check.